# Figure 3

In [1]:
import numpy as np
import pickle
from flax import nnx
from pathlib import Path
import string
from jax import numpy as jnp
import jax
import sys

# plot related
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

plt.rcParams["svg.fonttype"] = "none"

# model related
from nntp.utils.model import restore_model_from_checkpoint
from nntp.utils.plot import (
    reorder_legend_handles_row_major,
    find_experiments,
    load_experiments_parallel,
)

from common import (
    subtitle_fontsize,
    panel_indexing_fontsize,
    feature_size,
    output_size,
    CHANNEL,
    CHANNEL_NAME_MAPPING,
    COLORS,
)

src_path = (Path.cwd().parent / "src").resolve()
sys.path.insert(0, str(src_path))
from CoSynRNNModel.CoSynRNNModel import CoSynRNN

In [4]:
def get_experiment_tasks(name):
    if name == "frdmmd":
        return [
            "fdgo-ry",
            "fdanti-ry",
            "reactgo-ry",
            "reactanti-ry",
            "delaygo-ry",
            "delayanti-ry",
            "multidm-ry",
            "multidelaydm-ry",
            "free",
        ]
    elif name == "mdmdrf":
        return [
            "multidelaydm-ry",
            "multidm-ry",
            "delaygo-ry",
            "delayanti-ry",
            "reactgo-ry",
            "reactanti-ry",
            "fdgo-ry",
            "fdanti-ry",
            "free",
        ]


# create path
out_path = Path("./output")
out_path.mkdir(parents=True, exist_ok=True)
root_path = Path("..").expanduser().resolve() / "figures/experiments"
experiments = find_experiments(root_path, get_experiment_tasks)
for path in experiments:
    print(path)


def load_experiment(_, path):
    config = pickle.load(open(path / "metadata_config.blob", "rb"))
    model = restore_model_from_checkpoint(
        path,
        CoSynRNN(nnx.Rngs(42), feature_size, output_size, config),
        800,
    )

    # ================= extra neurons =================
    W_rec_active_synapse = jnp.abs(model.W_rec) > model.threshold

    recurrent_free_neurons = (
        (W_rec_active_synapse.sum(axis=0) == 0)
        & (W_rec_active_synapse.sum(axis=1) == 0)
        & model.trainable_mask
    )

    cue = int(jax.device_get(model.cue.get_value()))

    readout_free_neurons = (
        jnp.sum(
            jnp.abs(model.W_out * model.modulation_activation(model.W_mask[cue]))
            > model.threshold,
            axis=-1,
        )
        == 0
    )
    trainable_mask_value = recurrent_free_neurons & readout_free_neurons

    task_index = jnp.max(model.identity)
    identity_value = jnp.where(trainable_mask_value, task_index + 1, model.identity)
    model.identity = identity_value
    return model


loaded_experiments = load_experiments_parallel(experiments, load_experiment)

{'tasks': ['fdgo-ry', 'fdanti-ry', 'reactgo-ry', 'reactanti-ry', 'delaygo-ry', 'delayanti-ry', 'multidm-ry', 'multidelaydm-ry', 'free'], 'paths': [PosixPath('/allen/aind/scratch/ivan.y.gao/CoSyn-RNN/figures/experiments/frdmmd/25035270/1/72dc7938e5066ece'), PosixPath('/allen/aind/scratch/ivan.y.gao/CoSyn-RNN/figures/experiments/frdmmd/25035270/42/01737adfea12e65c'), PosixPath('/allen/aind/scratch/ivan.y.gao/CoSyn-RNN/figures/experiments/frdmmd/25035270/128/fabc6213f847706f'), PosixPath('/allen/aind/scratch/ivan.y.gao/CoSyn-RNN/figures/experiments/frdmmd/25035270/128128/d89bcd5943f241a8')]}
{'tasks': ['multidelaydm-ry', 'multidm-ry', 'delaygo-ry', 'delayanti-ry', 'reactgo-ry', 'reactanti-ry', 'fdgo-ry', 'fdanti-ry', 'free'], 'paths': [PosixPath('/allen/aind/scratch/ivan.y.gao/CoSyn-RNN/figures/experiments/mdmdrf/25035271/1/72dc7938e5066ece'), PosixPath('/allen/aind/scratch/ivan.y.gao/CoSyn-RNN/figures/experiments/mdmdrf/25035271/42/01737adfea12e65c'), PosixPath('/allen/aind/scratch/ivan.

E0809 19:21:52.732383  283959 cuda_executor.cc:1182] [0] Failed to allocate device memory of 29.64GiB (31824281600 bytes): RESOURCE_EXHAUSTED: : CUDA_ERROR_OUT_OF_MEMORY: out of memory
E0809 19:21:52.732463  283959 cuda_executor.cc:1182] [0] Failed to allocate device memory of 26.67GiB (28641853440 bytes): RESOURCE_EXHAUSTED: : CUDA_ERROR_OUT_OF_MEMORY: out of memory
E0809 19:21:52.732522  283959 cuda_executor.cc:1182] [0] Failed to allocate device memory of 24.01GiB (25777668096 bytes): RESOURCE_EXHAUSTED: : CUDA_ERROR_OUT_OF_MEMORY: out of memory
E0809 19:21:52.732572  283959 cuda_executor.cc:1182] [0] Failed to allocate device memory of 21.61GiB (23199899648 bytes): RESOURCE_EXHAUSTED: : CUDA_ERROR_OUT_OF_MEMORY: out of memory
E0809 19:21:52.732619  283959 cuda_executor.cc:1182] [0] Failed to allocate device memory of 19.45GiB (20879908864 bytes): RESOURCE_EXHAUSTED: : CUDA_ERROR_OUT_OF_MEMORY: out of memory
E0809 19:21:52.732667  283959 cuda_executor.cc:1182] [0] Failed to allocate

In [5]:
def plot_model_recurrent(ax, tasks, W_value, identity_value, threshold, show_cb):
    W_value = np.asarray(W_value)
    identity_value = np.asarray(identity_value).astype(int)

    active_mask = np.abs(W_value) > threshold

    # identity stores ordered task IDs: 0, 1, 2, ...
    task_ids = np.unique(identity_value)
    task_ids = task_ids[task_ids >= 0]

    orders = []
    block_labels = []

    for task_id in task_ids:
        task_indices = np.where(identity_value == task_id)[0]

        if len(task_indices) == 0:
            continue

        # Active connections within this block
        block_active = active_mask[np.ix_(task_indices, task_indices)]

        # Sort neurons by active connection count
        block_row_count = active_mask[task_indices, :].sum(axis=1)
        block_col_count = block_active.sum(axis=0)
        block_score = block_row_count + block_col_count

        task_order = task_indices[np.argsort(-block_score)]

        orders.append(task_order)
        block_labels.append(task_id)

    if not orders:
        raise ValueError("No valid task IDs were found in identity_value.")

    order = np.concatenate(orders)

    W_sorted = W_value[np.ix_(order, order)]
    W_display = np.clip(W_sorted, -threshold, threshold)

    # ==============================================
    # Heatmap
    # ==============================================
    image = ax.imshow(
        W_display,
        aspect="equal",
        cmap="coolwarm",
        norm=TwoSlopeNorm(
            vmin=-threshold,
            vcenter=0.0,
            vmax=threshold,
        ),
        interpolation="nearest",
    )
    # ==============================================
    # Block information
    # ==============================================
    block_sizes = np.asarray([len(task_order) for task_order in orders], dtype=int)
    block_starts = np.concatenate(
        [
            [0],
            np.cumsum(block_sizes)[:-1],
        ]
    )

    boundaries = np.cumsum(block_sizes)[:-1]

    # ==============================================
    # Block boundaries on the heatmap
    # ==============================================
    for boundary in boundaries:
        ax.axhline(boundary - 0.5, color="black", linewidth=0.5)
        ax.axvline(boundary - 0.5, color="black", linewidth=0.5)

    # ==============================================
    # Colors for different identity blocks
    # ==============================================
    # Full task names, used to look up colors
    block_task_names = [tasks[int(task_id)] for task_id in block_labels]

    # Use your predefined task colors
    block_colors = [COLORS[task_name] for task_name in block_task_names]
    num_blocks = len(block_labels)
    block_cmap = ListedColormap(block_colors)

    # Expand block index to one value per sorted neuron
    block_index_per_neuron = np.empty(
        len(order),
        dtype=int,
    )

    for block_index, (start, size) in enumerate(zip(block_starts, block_sizes)):
        block_index_per_neuron[start : start + size] = block_index

    # ==============================================
    # Top horizontal block strip
    # ==============================================
    ax_top_strip = ax.inset_axes(
        [
            0.0,  # x position
            1.01,  # y position
            1.0,  # width
            0.02,  # height
        ]
    )

    ax_top_strip.imshow(
        block_index_per_neuron[np.newaxis, :],
        aspect="auto",
        cmap=block_cmap,
        vmin=-0.5,
        vmax=num_blocks - 0.5,
        interpolation="nearest",
    )

    ax_top_strip.set_xticks([])
    ax_top_strip.set_yticks([])

    for spine in ax_top_strip.spines.values():
        spine.set_visible(False)

    # ==============================================
    # Left vertical task block strip
    # ==============================================
    ax_left_strip = ax.inset_axes(
        [
            -0.03,  # x position
            0.0,  # y position
            0.02,  # width
            1.0,  # height
        ]
    )

    ax_left_strip.imshow(
        block_index_per_neuron[:, np.newaxis],
        aspect="auto",
        cmap=block_cmap,
        vmin=-0.5,
        vmax=num_blocks - 0.5,
        interpolation="nearest",
    )

    ax_left_strip.set_xticks([])
    ax_left_strip.set_yticks([])

    for spine in ax_left_strip.spines.values():
        spine.set_visible(False)

    # ==============================================
    # Weight colorbar
    # ==============================================
    if show_cb:
        ax_cbar = ax.inset_axes(
            [
                1.010,  # x position
                0.0,  # y position
                0.025,  # width
                1.0,  # height
            ]
        )
        cbar = ax.figure.colorbar(image, cax=ax_cbar)
        cbar.set_label(r"Weight ($\times 10^{-4}$)", labelpad=-10)
        cbar.set_ticks([-threshold, threshold])
        cbar.ax.yaxis.set_major_formatter(
            FuncFormatter(lambda value, position: f"{value / 1e-4:g}")
        )
        cbar.outline.set_visible(False)

    # ==============================================
    # Block legend
    # ==============================================
    ax.legend(
        handles=reorder_legend_handles_row_major(
            [
                Patch(
                    facecolor=COLORS[task],
                    edgecolor="none",
                    label=CHANNEL_NAME_MAPPING[task.removesuffix("-ry")],
                )
                for task in tasks
                if "free" not in task.lower()
            ],
            ncol=4,
        ),
        loc="upper center",
        bbox_to_anchor=(0.5, 1.12),
        ncol=4,
        frameon=False,
    )

    # ==============================================
    # Labels
    # ==============================================
    ax.set_xlabel("Presynaptic Neurons")
    if not show_cb:
        ax.set_ylabel("Postsynaptic Neurons", labelpad=22)
    ax.tick_params(
        axis="y",
        which="both",
        left=False,
        labelleft=False,
    )
    return order

In [6]:
def plot_modulated_readout(
    axes,
    Ws,
    channel,
    tasks,
    threshold,
    show_ylabel=False,
):
    Ws = np.asarray(Ws)
    channel = np.asarray(channel, dtype=bool)
    axes = np.asarray(axes, dtype=object).ravel()

    # Map canonical task name to channel index
    channel_id_by_name = {
        task_name: channel_id for channel_id, task_name in enumerate(CHANNEL)
    }
    # Normalize task names while preserving the order in `tasks`
    ordered_task_names = [
        task.removesuffix("-ry")
        for task in tasks
        if task.removesuffix("-ry") in channel_id_by_name
    ]

    # Keep only channels that are active, in `tasks` order
    selected_names = [
        task_name
        for task_name in ordered_task_names
        if channel[channel_id_by_name[task_name]]
    ]

    selected_ids = np.asarray(
        [channel_id_by_name[task_name] for task_name in selected_names],
        dtype=int,
    )

    selected_Ws = Ws[selected_ids]

    if len(axes) < len(selected_ids):
        raise ValueError(
            f"Received {len(axes)} axes, "
            f"but {len(selected_ids)} channels are active."
        )

    images = []
    num_selected = len(selected_ids)

    for i, (ax, task_name, W) in enumerate(zip(axes, selected_names, selected_Ws)):
        W_display = np.clip(W.T, -threshold, threshold)

        image = ax.imshow(
            W_display,
            aspect="auto",
            cmap="coolwarm",
            norm=TwoSlopeNorm(
                vmin=-threshold,
                vcenter=0.0,
                vmax=threshold,
            ),
            interpolation="nearest",
        )
        ax.set_yticks([0, output_size])
        ax.set_title(CHANNEL_NAME_MAPPING[task_name])

        # Only the last active row shows x-axis labels
        is_bottom = i == num_selected - 1

        if is_bottom:
            ax.set_xlabel("Neurons")
            ax.tick_params(
                axis="x",
                which="both",
                bottom=True,
                labelbottom=True,
            )
        else:
            ax.set_xlabel("")
            ax.tick_params(
                axis="x",
                which="both",
                bottom=False,
                labelbottom=False,
            )

        # Only the left column shows y-axis labels
        if show_ylabel:
            ax.tick_params(
                axis="y",
                which="both",
                left=True,
                labelleft=True,
            )
        else:
            ax.tick_params(
                axis="y",
                which="both",
                left=False,
                labelleft=False,
            )

        images.append(image)

    for ax in axes[num_selected:]:
        ax.axis("off")

    return images

In [7]:
def plot_experiment(
    ax_top, ax_excitation, ax_readout, axes_bottom, tasks, model, show_ylabel, show_cb
):
    threshold = model.threshold
    W_out = model.W_out

    # rec
    order = plot_model_recurrent(
        ax_top, tasks, model.W_rec.T, model.identity, threshold, show_cb
    )

    # excitation
    excitation = model.excitation[order][None, :]
    image = ax_excitation.imshow(excitation, cmap="viridis", aspect="auto")
    ax_excitation.set_title("Gains")
    ax_excitation.set_xlabel("")
    ax_excitation.set_yticks([])
    ax_excitation.tick_params(
        axis="x",
        which="both",
        bottom=False,
        labelbottom=False,
    )
    if show_cb:
        ax_excitation_cbar = inset_axes(
            ax_excitation,
            width="2.5%",
            height="100%",
            loc="center left",
            bbox_to_anchor=(1.01, 0.0, 1.0, 1.0),
            bbox_transform=ax_excitation.transAxes,
            borderpad=0,
        )
        cbar = ax_excitation.figure.colorbar(image, cax=ax_excitation_cbar)
        cbar.outline.set_visible(False)

    # readout
    readout = W_out[order, :][None, :].T
    limit = np.max(np.abs(readout))
    image = ax_readout.imshow(
        readout,
        cmap="coolwarm",
        aspect="auto",
        norm=TwoSlopeNorm(
            vmin=-limit,
            vcenter=0.0,
            vmax=limit,
        ),
    )
    ax_readout.set_title(r"$W_{\text{out}}$")
    ax_readout.set_xlabel("")
    ax_readout.set_yticks([0, output_size])
    ax_readout.tick_params(
        axis="x",
        which="both",
        bottom=False,
        labelbottom=False,
    )
    if show_cb:
        ax_readout_cbar = inset_axes(
            ax_readout,
            width="2.5%",
            height="100%",
            loc="center left",
            bbox_to_anchor=(1.01, 0.0, 1.0, 1.0),
            bbox_transform=ax_readout.transAxes,
            borderpad=0,
        )
        cbar = ax_readout.figure.colorbar(image, cax=ax_readout_cbar)
        cbar.set_ticks([-limit, limit])
        cbar.set_ticklabels([f"{-limit:.1f}", f"{limit:.1f}"])
        cbar.outline.set_visible(False)

    if show_ylabel:
        ax_readout.set_ylabel("Output")
        ax_readout.tick_params(
            axis="y",
            which="both",
            left=True,
            labelleft=True,
        )
    else:
        ax_readout.set_ylabel("")
        ax_readout.tick_params(
            axis="y",
            which="both",
            left=False,
            labelleft=False,
        )

    modulated_readout = W_out[None, :, :] * model.modulation_activation(model.W_mask)
    images = plot_modulated_readout(
        axes_bottom,
        modulated_readout[:, order, :],
        model.channel,
        tasks,
        threshold,
        show_ylabel,
    )

    if show_cb:
        # 使用当前这一列所属的 figure 和 axes
        fig = axes_bottom[0].figure
        fig.canvas.draw()

        top_box = axes_bottom[0].get_position()
        bottom_box = axes_bottom[-1].get_position()

        gap = 0.010 * top_box.width
        cbar_width = 0.025 * top_box.width

        ax_bottom_cbar = fig.add_axes(
            [
                top_box.x1 + gap,
                bottom_box.y0,
                cbar_width,
                top_box.y1 - bottom_box.y0,
            ]
        )

        bottom_cbar = fig.colorbar(images[0], cax=ax_bottom_cbar)
        bottom_cbar.set_ticks([-threshold, threshold])
        bottom_cbar.ax.yaxis.set_major_formatter(
            FuncFormatter(lambda value, position: f"{value / 1e-4:g}")
        )
        bottom_cbar.set_label(r"Weight ($\times 10^{-4}$)", labelpad=-10)
        bottom_cbar.outline.set_visible(False)

    return images

In [8]:
for index, seed in enumerate([1, 42, 128, 128128]):
    # for index, seed in enumerate([1]):
    fig = plt.figure(figsize=(17, 18))

    gs = fig.add_gridspec(
        nrows=12,
        ncols=2,
        width_ratios=[1, 1],
        height_ratios=[8, 0.1] + [0.5] * 10,
        hspace=0.35,
        wspace=0.05,
    )

    # 第一行：两个大图
    ax_top_left = fig.add_subplot(gs[0, 0])
    ax_top_right = fig.add_subplot(gs[0, 1])

    # 第二行：第一组 middle
    ax_middle_left_excitation = fig.add_subplot(gs[2, 0])
    ax_middle_right_excitation = fig.add_subplot(
        gs[2, 1],
        sharex=ax_middle_left_excitation,
        sharey=ax_middle_left_excitation,
    )

    # 第三行：第二组 middle
    ax_middle_left_readout = fig.add_subplot(gs[3, 0])
    ax_middle_right_readout = fig.add_subplot(
        gs[3, 1],
        sharex=ax_middle_left_readout,
        sharey=ax_middle_left_readout,
    )

    # 后面八行
    axes_bottom_left = [
        fig.add_subplot(
            gs[row, 0],
            sharex=ax_middle_left_readout,
            sharey=ax_middle_left_readout,
        )
        for row in range(4, 12)
    ]

    axes_bottom_right = [
        fig.add_subplot(
            gs[row, 1],
            sharex=ax_middle_left_readout,
            sharey=ax_middle_left_readout,
        )
        for row in range(4, 12)
    ]

    plot_experiment(
        ax_top_left,
        ax_middle_left_excitation,
        ax_middle_left_readout,
        axes_bottom_left,
        loaded_experiments[0]["tasks"],
        loaded_experiments[0]["data"][index],
        show_ylabel=True,
        show_cb=False,
    )

    plot_experiment(
        ax_top_right,
        ax_middle_right_excitation,
        ax_middle_right_readout,
        axes_bottom_right,
        loaded_experiments[1]["tasks"],
        loaded_experiments[1]["data"][index],
        show_ylabel=False,
        show_cb=True,
    )

    fig.text(
        0.30,
        0.93,
        "Forward Order",
        ha="center",
        va="center",
        fontsize=14,
        fontweight="bold",
    )
    fig.text(
        0.70,
        0.93,
        "Reverse Order",
        ha="center",
        va="center",
        fontsize=14,
        fontweight="bold",
    )
    fig.text(
        0.10,
        0.30,
        "Output",
        rotation=90,
        va="center",
        ha="center",
    )

    panel_axes = [
        ax_top_left,
        ax_top_right,
        ax_middle_left_excitation,
        ax_middle_right_excitation,
        ax_middle_left_readout,
        ax_middle_right_readout,
    ]

    for ax_left, ax_right in zip(axes_bottom_left, axes_bottom_right):
        panel_axes.extend([ax_left, ax_right])

    for i, (label, ax) in enumerate(zip(string.ascii_lowercase, panel_axes)):
        if i < 2:
            x, y = -0.05, 1.05
        else:
            x, y = 0.00, 1.05

        ax.text(
            x,
            y,
            f"{label}",
            transform=ax.transAxes,
            ha="left",
            va="bottom",
            fontsize=12,
            fontweight="bold",
            clip_on=False,
        )

    filename_png = (
        "Figure3.png" if seed == 1 else f"SupplementaryFigure2_seed_{seed}.png"
    )
    # filename_svg = (
    #     "Figure3.svg" if seed == 1 else f"SupplementaryFigure2_seed_{seed}.svg"
    # )

    fig.savefig(
        out_path / filename_png,
        dpi=300,
        bbox_inches="tight",
    )
    # fig.savefig(
    #     out_path / filename_svg,
    #     bbox_inches="tight",
    # )
    plt.close(fig)